### 🎯 [미션] 주석이 달린 줄의 None 또는 _______ 부분을 채워 코드를 완성하세요.

각 셀을 순서대로 실행(Shift + Enter)해야 합니다.

---

# 🚗 자율주행 신호 해석 시스템

이 노트북에서는 학습된 모델을 사용하여 **교통 신호를 탐지하고 자율주행 동작을 결정**하는 시스템을 구현합니다.

**시스템 기능:**
1. 이미지에서 교통 신호 탐지 (3개 클래스)
2. 신호에 따른 자율주행 동작 결정
3. 결과 시각화 및 동작 안내

**신호별 동작:**
| 신호 | 동작 | 설명 |
|------|------|------|
| 🟢 green | GO | 진행하세요 |
| 🔴 red | STOP | 정지하세요 |
| 🟡 yellow | CAUTION | 서행하세요 |

---
#### 1️⃣ 필요한 라이브러리 설치 및 불러오기

In [ ]:
# 필요한 라이브러리 설치 (최초 1회만 실행)
# - ultralytics: YOLO 모델 사용을 위한 라이브러리
# - koreanize-matplotlib: 그래프에서 한글 폰트 지원
!pip install ultralytics koreanize-matplotlib

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import koreanize_matplotlib
from PIL import Image as PILImage

from ultralytics import YOLO

print("✅ 라이브러리 로딩 완료!")

---
#### 2️⃣ 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd "/content/drive/MyDrive/2026_AI_Advanced_Study-week4/4차시/05_traffic_light/code"

---
#### 3️⃣ 학습된 모델 불러오기

In [ ]:
# 학습된 모델의 경로 설정
model_path = './runs/detect/train/weights/best.pt'

# 모델 로드
model = YOLO(model_path)

print(f"✅ 모델 로드 완료: {model_path}")
print(f"   클래스: {model.names}")

---
#### 4️⃣ 자율주행 동작 매핑 함수

탐지된 신호에 따라 자율주행 차량이 취해야 할 동작을 결정합니다.

**클래스 ID와 동작:**
- 0: red → STOP (정지)
- 1: green → GO (진행)
- 2: yellow → CAUTION (서행)

In [ ]:
def get_driving_action(signal_class):
    """
    탐지된 신호에 따른 자율주행 동작을 반환합니다.
    
    Args:
        signal_class: 신호 클래스 ID (0-2)
        
    Returns:
        tuple: (동작, 메시지, 색상)
    """
    actions = {
        0: ('STOP', '🔴 정지하세요', (255, 0, 0)),      # red
        1: ('GO', '🟢 진행하세요', (0, 255, 0)),        # green
        2: ('CAUTION', '🟡 서행하세요', (255, 255, 0))  # yellow
    }
    return actions.get(signal_class, ('UNKNOWN', '❓ 신호 불명', (128, 128, 128)))

print("✅ get_driving_action 함수 정의 완료!")

---
#### 5️⃣ 신호 분석 함수 구현

이미지에서 탐지된 신호를 분석하여 주요 동작을 결정합니다.

**결정 로직: 이미지 중앙에 가장 가까운 신호등 선택**
- 여러 신호등이 탐지되면, 내 앞(이미지 중앙)에 있는 신호등을 기준으로 동작 결정
- 좌우 끝에 있는 신호등은 다른 차선의 신호일 가능성이 높음

**Detection 결과 구조:**
- `result.boxes`: 탐지된 모든 바운딩 박스
- `box.cls`: 클래스 ID
- `box.conf`: 신뢰도
- `box.xyxy`: 좌표 (x1, y1, x2, y2)

In [ ]:
def analyze_signal(result, image_width=640):
    """
    YOLO 예측 결과를 분석하여 신호 정보를 반환합니다.
    이미지 중앙에 가장 가까운 신호등을 기준으로 동작을 결정합니다.
    
    Args:
        result: YOLO 예측 결과 (단일 이미지)
        image_width: 이미지 너비 (기본값 640)
        
    Returns:
        dict: 신호 분석 결과
    """
    # 🎯 [미션] Detection 결과에서 바운딩 박스를 가져오세요.
    # 힌트: Classification은 probs, Detection은?
    boxes = result._______
    
    # 이미지 중앙 x좌표
    center_x = image_width / 2
    
    # 탐지된 신호 목록
    detected_signals = []
    
    # 각 바운딩 박스 분석
    for box in boxes:
        # 🎯 [미션] 바운딩 박스에서 클래스 ID를 추출하세요.
        # 힌트: box의 어떤 속성에 클래스 정보가 있을까요?
        cls_id = int(box._______[0])
        conf = float(box.conf[0])
        
        # 바운딩 박스 중심 x좌표 계산
        x1, y1, x2, y2 = box.xyxy[0]
        box_center_x = (x1 + x2) / 2
        
        # 이미지 중앙에서의 거리 계산
        distance_from_center = abs(box_center_x - center_x)
        
        # 🎯 [미션] 클래스 ID로 자율주행 동작을 가져오세요.
        action, message, color = _______
        
        detected_signals.append({
            'cls_id': cls_id,
            'action': action,
            'message': message,
            'color': color,
            'confidence': conf,
            'center_x': float(box_center_x),
            'distance_from_center': float(distance_from_center)
        })
    
    # 주요 동작 결정: 이미지 중앙에 가장 가까운 신호등 선택
    if detected_signals:
        # 중앙에서 가장 가까운 신호등 선택
        primary = min(detected_signals, key=lambda x: x['distance_from_center'])
    else:
        primary = {
            'cls_id': -1,
            'action': 'NO_SIGNAL',
            'message': '❓ 신호 없음',
            'color': (128, 128, 128),
            'confidence': 0,
            'center_x': 0,
            'distance_from_center': 0
        }
    
    return {
        'all_signals': detected_signals,
        'primary': primary,
        'total_count': len(detected_signals)
    }

print("✅ analyze_signal 함수 정의 완료!")

---
#### 6️⃣ 신호 해석 시각화 함수

In [ ]:
def visualize_signal(image_path, model):
    """
    이미지에서 신호를 탐지하고 자율주행 동작을 시각화합니다.
    """
    # 🎯 [미션] 이미지 예측을 수행하세요.
    # 힌트: model의 어떤 메서드로 예측을 수행할까요?
    results = model._______(image_path, verbose=False)
    result = results[0]
    
    # 이미지 로드
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    # 신호 분석 (이미지 너비 전달)
    analysis = analyze_signal(result, image_width=w)
    primary = analysis['primary']
    
    # 바운딩 박스 그리기
    for box in result.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        
        # 신호별 색상
        action, message, color = get_driving_action(cls_id)
        cls_name = model.names[cls_id]
        
        # 박스 그리기
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
        label = f"{cls_name} {conf:.0%}"
        cv2.putText(img, label, (x1, y1-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    
    # 이미지 중앙선 그리기 (시각적 확인용)
    center_x = w // 2
    cv2.line(img, (center_x, 0), (center_x, h), (255, 255, 255), 1)
    
    # 시각화
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # 왼쪽: 탐지 결과
    axes[0].imshow(img)
    axes[0].set_title(f'Signal Detection - {primary["action"]}', fontsize=14)
    axes[0].axis('off')
    
    # 오른쪽: 분석 결과
    axes[1].set_xlim(0, 10)
    axes[1].set_ylim(0, 10)
    axes[1].axis('off')
    
    # 상태 배너
    action_colors = {
        'GO': 'green',
        'STOP': 'red',
        'CAUTION': 'orange',
        'NO_SIGNAL': 'gray',
        'UNKNOWN': 'gray'
    }
    status_color = action_colors.get(primary['action'], 'gray')
    
    axes[1].text(5, 9, '🚗 자율주행 신호 해석', fontsize=18, ha='center', fontweight='bold')
    axes[1].text(5, 7.5, primary['action'], fontsize=28, ha='center', fontweight='bold', color=status_color)
    axes[1].text(5, 6, primary['message'], fontsize=14, ha='center')
    
    # 탐지된 신호 목록 (중앙에서의 거리 순으로 정렬)
    sorted_signals = sorted(analysis['all_signals'], key=lambda x: x['distance_from_center'])
    axes[1].text(1, 4.5, f"📊 탐지된 신호: {analysis['total_count']}개 (중앙 가까운 순)", fontsize=12, fontweight='bold')
    
    y_pos = 3.5
    for i, sig in enumerate(sorted_signals):
        sig_color = action_colors.get(sig['action'], 'gray')
        marker = "→" if i == 0 else "  "
        axes[1].text(1.2, y_pos, f"{marker} {sig['action']}: {sig['confidence']:.1%}", 
                     fontsize=12, color=sig_color, fontweight='bold' if i == 0 else 'normal')
        y_pos -= 0.7
    
    plt.tight_layout()
    plt.show()
    
    return analysis

print("✅ visualize_signal 함수 정의 완료!")

---
#### 7️⃣ 시스템 테스트

In [ ]:
import glob

# 데모 이미지 테스트
demo_images = glob.glob('../data/demo/*.[jJpP][pPnN][gG]')

print(f"📷 테스트할 이미지: {len(demo_images)}장\n")

for img_path in demo_images:
    print("=" * 60)
    print(f"📁 파일: {os.path.basename(img_path)}")
    print("=" * 60)
    
    analysis = visualize_signal(img_path, model)
    print()